# E2 - ToupTek photon-transfer calibration with the beta light

Photon-transfer calibration of the ToupTek MTR3CMOS09000KMA at the operating point of E4 (gain 23, HCG, black level 1000, -30 °C), with the beta light mounted directly in front of the sensor, followed by a dark ladder. Gives the conversion gain (e-/ADU), read noise and offset, and the dark current in e-/px/s, to settle the factor ~10 between the datasheet and what was measured so far.

How it runs:

1. **Exposure check** (light mounted): snap, look at the histogram, adjust `BRIGHT_EXPOSURE_MS`.
2. **PTC per gain** through ImSwitch's `ReadNoiseCalibrationController` (the FRAME Settings wizard):
   20 bright frames with the light mounted, 20 dark frames with the light removed and the camera
   capped, then the `cal_readnoise` fit. Results, figures and stacks are copied into the run folder.
   The notebook stops and asks you each time the light has to go on or off.
3. **Dark ladder** (light removed): dark frames at several exposures. The dark current is the slope
   of the dark level (and of the temporal variance) against exposure time.

The report adds a per-pixel temporal photon-transfer fit from the same stacks as a cross-check:
`cal_readnoise` has read about 20 % low on ideal synthetic data.

**Before you start**: nothing here moves the stage or switches a light. Run once with
`DRY_RUN = True` (3-frame stacks, short ladder, no waiting for you), then set it to `False`.

## 1 Connect

In [ ]:
from pathlib import Path
import biophoton as bp

# None = $IMSWITCH_API_URL, which ImSwitch sets for the notebooks it serves itself.
# From another computer use the address you open in the browser, e.g. "http://192.168.1.20/imswitch/api".
IMSWITCH_URL = None
client = bp.connect(IMSWITCH_URL)
print("Connected to", client.base_uri, "| detectors:", client.settingsManager.getDetectorNames())

## 2 Settings

In [ ]:
DETECTOR = client.settingsManager.getDetectorNames()[0]
RUN_NAME = "touptek_ptc"
DRY_RUN = True

CAMERA = {                   # ToupCamManager parameter names, applied in this order
    "exposure_mode": "manual",
    "pixel_format": "mono16",    # before the black level: its range depends on the bit depth
    "conversion_gain": "HCG",    # low read noise mode
    "low_noise": True,
    "gain": 23,                  # ImSwitch scale 0..23 maps linearly onto the native %; 23 = maximum
    "blacklevel": 1000,          # fixed offset instead of dark frames (also turns OB auto-adjust off)
    "target_temperature": -30,
}
GAINS = [23]                 # one PTC per gain; add 0 to see the conversion factor at minimum gain (native 100 % = 1x)
N_BRIGHT = N_DARK = 20
BRIGHT_EXPOSURE_MS = 1000    # set with the exposure check below
DARK_LADDER_S = [1, 10, 60, 300, 900]   # dark-only exposures; [] skips the ladder (~1.5 h with 4 frames each)
DARK_LADDER_FRAMES = 4
DOWNLOAD_STACKS = True       # copy both stacks into the run folder (~360 MB each at full frame) for the cross-check
DATASHEET_DARK_E_PX_S = 5e-4 # ToupTek figure for the IMX533 at -20 °C as read from the transcript; check David's sheet
COOLDOWN = {"tolerance_c": 1.0, "timeout_s": 30 * 60}
ENV_PERIOD_S = 30
OTHER_SERVERS = []
DATA_ROOT = Path("data")

if DRY_RUN:
    N_BRIGHT = N_DARK = 3
    DARK_LADDER_S, DARK_LADDER_FRAMES = [0.2, 1], 2

## 3 Run folder, camera settings, environment log, cooldown

In [ ]:
try:
    if acq.running:
        raise RuntimeError("An acquisition is still running in this kernel - stop it first (Stop cell)")
except NameError:
    pass

run = bp.Run("E2_touptek_ptc", RUN_NAME, DATA_ROOT)
readback = bp.apply_settings(client, DETECTOR, CAMERA)
run.log(f"camera read back: {readback}")
run.update_config(
    kind="ptc", experiment="E2", title="E2 - ToupTek photon-transfer calibration",
    description='Photon-transfer calibration of the ToupTek MTR3CMOS09000KMA at the operating point of E4 (gain 23, HCG, black level 1000, -30 °C), with the beta light mounted directly in front of the sensor, followed by a dark ladder. Gives the conversion gain (e-/ADU), read noise and offset, and the dark current in e-/px/s, to settle the factor ~10 between the datasheet and what was measured so far.',
    server=client.base_uri, detector=DETECTOR, dry_run=DRY_RUN, camera_settings=CAMERA,
    camera_readback=readback, gains=GAINS, n_bright=N_BRIGHT, n_dark=N_DARK,
    dark_ladder_s=DARK_LADDER_S, dark_ladder_frames=DARK_LADDER_FRAMES, dark_ladder_gain=GAINS[0],
    datasheet_dark_e_px_s=DATASHEET_DARK_E_PX_S, ptc_sessions=[])
run.save_json("camera_start.json", bp.parameters(client, DETECTOR)[1])

env = bp.EnvLogger(run.dir / "environment.csv",
                   bp.env_sources(client, DETECTOR, "toupcam", OTHER_SERVERS), ENV_PERIOD_S).start()
bp.wait_for_temperature(client, DETECTOR, CAMERA["target_temperature"], log=run.log, **COOLDOWN)

light = {"mounted": False}


def need_light(mounted):
    """Ask for the beta light to be mounted or removed, only when that changes."""
    if light["mounted"] != mounted:
        bp.prompt("MOUNT the beta light directly in front of the sensor" if mounted
                  else "REMOVE the beta light and cap the camera", DRY_RUN, run.log)
        light["mounted"] = mounted

## 4 Exposure check

Re-run with a different `BRIGHT_EXPOSURE_MS` until the brightest region sits near 70 % of full scale and nothing clips. If the camera delivers 14 bit in a 16-bit container the real top is lower than 65535: the frame max shows where it clips.

In [ ]:
need_light(True)
bp.apply_settings(client, DETECTOR, {"gain": GAINS[0]})
client.settingsManager.setDetectorExposureTime(DETECTOR, BRIGHT_EXPOSURE_MS)
rel = bp.snap_recording(client, f"{run.dir.name}_check_{BRIGHT_EXPOSURE_MS}ms")
run.log(f"exposure check at {BRIGHT_EXPOSURE_MS} ms: {rel}")
bp.show_exposure_check(client.recordingManager.downloadImage(rel))

## 5 Photon transfer per gain

Asks you to remove or mount the light when needed. Each gain is one ImSwitch session, also visible in FRAME Settings → Read Noise Calibration.

In [ ]:
sessions = []
for g in GAINS:
    bp.apply_settings(client, DETECTOR, {"gain": g})
    client.settingsManager.setDetectorExposureTime(DETECTOR, BRIGHT_EXPOSURE_MS)
    s = client.readnoiseManager.startSession(name=f"{RUN_NAME}_gain{g}", detector_name=DETECTOR,
                                             nBright=N_BRIGHT, nDark=N_DARK)["session"]
    run.log(f"gain {g}: session {s['sessionId']}")
    # whichever stack the light is ready for first: one swap per gain
    for kind in (("bright", "dark") if light["mounted"] else ("dark", "bright")):
        need_light(kind == "bright")
        run.log(f"gain {g}: {kind} stack")
        client.readnoiseManager.acquireStack(
            kind, on_progress=lambda p: print("\r" + str(p.get("message", "")), end="   "))
        print()
    result = client.readnoiseManager.computeCalibration(s["sessionId"])
    if result.get("status") != "ok":
        raise RuntimeError(f"computeCalibration failed: {result}")
    r = result["result"]
    run.log(f"gain {g}: {r['gain']:.4g} e-/ADU, read noise {r['readnoise']:.3g} e-, "
            f"offset {r['offset']:.1f} ADU")
    local = bp.fetch_ptc_session(run, client, result, DOWNLOAD_STACKS)
    sessions.append({"gain": g, "exposure_ms": BRIGHT_EXPOSURE_MS, "sessionId": s["sessionId"],
                     "dir": local.relative_to(run.dir).as_posix()})
    run.update_config(ptc_sessions=sessions)

## 6 Dark ladder

Light removed, camera capped. Runs in the background; interrupting the dashboard (■) does not stop it, `acq.stop()` does.

In [ ]:
need_light(False)
bp.apply_settings(client, DETECTOR, {"gain": GAINS[0]})
STEPS = [{"label": f"dark_{t:g}s", "exposure_ms": t * 1000, "n_frames": DARK_LADDER_FRAMES}
         for t in DARK_LADDER_S]
run.update_config(steps=STEPS)
acq = bp.Acquisition(run, client, DETECTOR, STEPS, backend="recording", env=env,
                     env_label="toupcam").start()
bp.dashboard(run.dir, acq, refresh_s=15)

## Stop

Interrupt the dashboard cell first (■): the kernel runs one cell at a time. This cancels the running
exposure on the server. Frames already saved stay on disk.

In [ ]:
acq.stop()
print(acq.state)

## Finish

Stops the environment log and stores the camera state at the end. Run it after the acquisition has
finished or was stopped.

In [ ]:
env.stop()
run.save_json("camera_end.json", bp.parameters(client, DETECTOR)[1])
run.log("run finished")

## Analysis and report

Reads only the run folder; point `RUN_DIR` at an older run to redraw it. The summary table compares the controller's `cal_readnoise` result with the temporal fit, and the dark ladder gives the dark current in e-/px/s, from the dark level and from the variance, with its ratio to the datasheet value.

In [ ]:
RUN_DIR = run.dir           # or e.g. Path("data/E2_touptek_ptc/<run folder>")
summary = bp.redraw(RUN_DIR, show=True)
summary

The run folder now holds `README.md` with this summary and the figures, and `report.ipynb`, which
redraws them from that folder alone (it imports the copy of `biophoton.py` stored with the run).